# Задание 4. Рекомендательная система на основе SVD

## Постановка задачи

Необходимо разработать рекомендательную систему для датасета Goodbooks-10k Extended на основе метода Singular Value Decomposition.

В качестве входных данных используются оценки пользователей книгам. Система должна восстановить скрытые факторы предпочтений пользователей и книг, после чего сформировать персональные рекомендации для каждого пользователя.

Основной гиперпараметр модели — размерность разложения n_components.

Для оценки качества будут рассчитаны RMSE, MAE, Precision@K, Recall@K, NDCG@K и HitRate@K. Для ранжирующих метрик будут рассмотрены K = 5, 10 и 20. Сравнение с предыдущим User-Based + Item-Based KNN будет выполнено при K = 10.

## Структура и содержание данных

Используется расширенная версия Goodbooks-10k.

Основные файлы:

ratings.csv содержит user_id, book_id и rating.

books_enriched.csv содержит информацию о книгах. Среди дополнительных признаков присутствуют title, authors, description, pages, publishDate, genres и language_code.

book_tags.csv и tags.csv содержат информацию о тегах и пользовательских полках книг.

Для SVD используются ratings.csv и books_enriched.csv. История взаимодействий задаётся оценками от 1 до 5. Книжные признаки не входят непосредственно в матрицу SVD, но используются для понятного отображения результатов рекомендаций.

Источник датасета: https://github.com/malcolmosh/goodbooks-10k-extended

По описанию репозитория датасет содержит 10 000 книг, 53 424 пользователей и около шести миллионов оценок. В расширенной версии описание имеется для 9943 книг.

In [1]:
import time
import math
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')

DATA_DIR = 'goodbooks-10k-extended'
GLOBAL_PARAMS = {
    'TEST_SIZE': 0.20,
    'RANDOM_STATE': 42,
    'TOP_K_VALUES': [5, 10, 20],
    'SVD_COMPONENTS': [10, 20, 50, 100, 200],
    'RELEVANCE_THRESHOLD': 4,
    'N_EVAL_USERS': 200
}

## Загрузка и анализ данных

Сначала загружаются оценки пользователей и метаданные книг. Проверяется количество пользователей, книг и взаимодействий, разреженность матрицы, средняя оценка и распределение оценок.

In [2]:
def load_data():
    ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv', usecols=['user_id', 'book_id', 'rating'])
    books = pd.read_csv(f'{DATA_DIR}/books_enriched.csv')
    if 'book_id' not in books.columns: books = books.rename(columns={books.columns[0]: 'book_id'})
    ratings = ratings.dropna(subset=['user_id', 'book_id', 'rating']).copy()
    books = books.dropna(subset=['book_id']).copy()
    ratings[['user_id', 'book_id']] = ratings[['user_id', 'book_id']].astype(np.int32)
    ratings['rating'] = ratings['rating'].astype(np.float32)
    books['book_id'] = books['book_id'].astype(np.int32)
    return ratings, books

ratings_df, books_df = load_data()

n_users = ratings_df['user_id'].nunique()
n_books = ratings_df['book_id'].nunique()
n_ratings = len(ratings_df)
sparsity = (1 - n_ratings / (n_users * n_books)) * 100

print('=' * 60)
print('АНАЛИЗ GOODBOOKS-10K EXTENDED')
print('=' * 60)
print(f'Уникальных пользователей: {n_users:,}')
print(f'Уникальных книг:          {n_books:,}')
print(f'Всего оценок:             {n_ratings:,}')
print(f'Разреженность матрицы:    {sparsity:.2f}%')
print(f'Средняя оценка:           {ratings_df["rating"].mean():.3f}')
print(f'Медианная оценка:         {ratings_df["rating"].median():.1f}')
print('=' * 60)
display(ratings_df['rating'].value_counts().sort_index().rename('count').to_frame())

book_cols = [c for c in ['title', 'authors', 'description', 'genres', 'pages', 'publishDate', 'language_code'] if c in books_df.columns]
display(books_df[book_cols].head())

АНАЛИЗ GOODBOOKS-10K EXTENDED
Уникальных пользователей: 53,424
Уникальных книг:          10,000
Всего оценок:             5,976,479
Разреженность матрицы:    98.88%
Средняя оценка:           3.920
Медианная оценка:         4.0


,count
rating,
1.0,124195
2.0,359257
3.0,1370916
4.0,2139018
5.0,1983093


,title,authors,description,genres,pages,publishDate,language_code
0,"The Hunger Games (The Hunger Games, #1)",['Suzanne Collins'],WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,"['young-adult', 'fiction', 'fantasy', 'science...",374.0,09/14/08,eng
1,Harry Potter and the Sorcerer's Stone (Harry P...,"['J.K. Rowling', 'Mary GrandPré']",Harry Potter's life is miserable. His parents ...,"['fantasy', 'fiction', 'young-adult', 'classics']",309.0,11/01/03,eng
2,"Twilight (Twilight, #1)",['Stephenie Meyer'],About three things I was absolutely positive.\...,"['young-adult', 'fantasy', 'romance', 'fiction...",501.0,09/06/06,eng
3,To Kill a Mockingbird,['Harper Lee'],The unforgettable novel of a childhood in a sl...,"['classics', 'fiction', 'historical-fiction', ...",324.0,05/23/06,eng
4,The Great Gatsby,['F. Scott Fitzgerald'],Alternate Cover Edition ISBN: 0743273567 (ISBN...,"['classics', 'fiction', 'historical-fiction', ...",200.0,09/28/04,eng


## Подготовка обучающей и тестовой выборок

Используется случайное разбиение 80 процентов на обучение и 20 процентов на тестирование с random_state = 42. Такая схема соответствует предыдущему эксперименту KNN и позволяет сопоставлять результаты.

В тестовой выборке релевантными считаются книги с оценкой 4 или 5.

In [3]:
train_df, test_df = train_test_split(ratings_df, test_size=GLOBAL_PARAMS['TEST_SIZE'], random_state=GLOBAL_PARAMS['RANDOM_STATE'], stratify=None)

user_ids = pd.Index(np.sort(train_df['user_id'].unique()))
book_ids = pd.Index(np.sort(train_df['book_id'].unique()))
user_to_idx = pd.Series(np.arange(len(user_ids)), index=user_ids)
book_to_idx = pd.Series(np.arange(len(book_ids)), index=book_ids)

rows = train_df['user_id'].map(user_to_idx).to_numpy()
cols = train_df['book_id'].map(book_to_idx).to_numpy()
vals = train_df['rating'].to_numpy(dtype=np.float32)

ui_matrix = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(book_ids)), dtype=np.float32)
ui_matrix.sum_duplicates()

user_sum = np.asarray(ui_matrix.sum(axis=1)).ravel()
user_count = np.asarray((ui_matrix != 0).sum(axis=1)).ravel()
user_mean = np.divide(user_sum, user_count, out=np.zeros_like(user_sum, dtype=np.float32), where=user_count > 0)

centered_data = ui_matrix.copy().astype(np.float32)
centered_rows = np.repeat(np.arange(centered_data.shape[0]), np.diff(centered_data.indptr))
centered_data.data = centered_data.data - user_mean[centered_rows]

print(f'Обучающая выборка: {len(train_df):,}')
print(f'Тестовая выборка:   {len(test_df):,}')
print(f'Матрица:            {ui_matrix.shape[0]:,} x {ui_matrix.shape[1]:,}')
print(f'Ненулевых значений: {ui_matrix.nnz:,}')

Обучающая выборка: 4,781,183
Тестовая выборка:   1,195,296
Матрица:            53,424 x 10,000
Ненулевых значений: 4,781,183


## Подход к реализации SVD

Для вычислений используется разреженное сингулярное разложение TruncatedSVD.

Перед разложением из каждой известной оценки пользователя вычитается его средняя оценка в обучающей выборке. Это уменьшает влияние того, что разные пользователи используют шкалу оценок по-разному.

Модель строит скрытое пространство заданной размерности n_components.

Для пользователя итоговая оценка книги рассчитывается как средняя оценка пользователя плюс восстановленное значение из SVD.

Книги, которые пользователь уже оценил в обучении, исключаются из выдачи. Оставшиеся книги сортируются по предсказанной оценке, после чего выбирается Top-K.

In [4]:
def fit_svd(n_components):
    model = TruncatedSVD(n_components=n_components, n_iter=10, random_state=GLOBAL_PARAMS['RANDOM_STATE'])
    start = time.time()
    user_factors = model.fit_transform(centered_data)
    elapsed = time.time() - start
    return model, user_factors, elapsed

def predict_for_users(model, user_factors, eval_positions):
    scores = user_factors[eval_positions] @ model.components_
    scores = scores + user_mean[eval_positions, np.newaxis]
    return scores

def get_test_relevant(test_df):
    result = {}
    relevant_df = test_df[test_df['rating'] >= GLOBAL_PARAMS['RELEVANCE_THRESHOLD']]
    for uid, group in relevant_df.groupby('user_id'):
        result[uid] = set(group['book_id'])
    return result

test_relevant = get_test_relevant(test_df)

np.random.seed(GLOBAL_PARAMS['RANDOM_STATE'])
candidate_eval_users = np.array([uid for uid in user_ids if uid in test_relevant])
sample_eval_users = np.random.choice(candidate_eval_users, size=min(GLOBAL_PARAMS['N_EVAL_USERS'], len(candidate_eval_users)), replace=False)
sample_eval_positions = np.array([int(user_to_idx.loc[uid]) for uid in sample_eval_users])

print(f'Пользователей для ranking-оценки: {len(sample_eval_users)}')

Пользователей для ranking-оценки: 200


## Метрики качества

RMSE показывает среднюю квадратичную ошибку предсказания рейтинга. Чем меньше значение, тем лучше.

MAE показывает среднюю абсолютную ошибку предсказания рейтинга. Чем меньше значение, тем лучше.

Precision@K показывает долю релевантных книг среди первых K рекомендаций. Чем больше значение, тем лучше.

Recall@K показывает долю найденных релевантных книг из всех релевантных книг пользователя в тесте. Чем больше значение, тем лучше.

HitRate@K показывает долю пользователей, для которых хотя бы одна релевантная книга попала в первые K рекомендаций. Чем больше значение, тем лучше.

NDCG@K учитывает не только наличие релевантных книг, но и их положение в списке. Чем больше значение, тем лучше.

In [5]:
def evaluate_model(model, user_factors, components, k_values=(5, 10, 20)):
    eval_positions = sample_eval_positions
    eval_users = sample_eval_users
    scores = predict_for_users(model, user_factors, eval_positions)
    train_csr = ui_matrix[eval_positions]
    scores = scores.copy()
    for i in range(scores.shape[0]):
        scores[i, train_csr[i].indices] = -np.inf

    test_pairs = test_df[test_df['user_id'].isin(eval_users)][['user_id', 'book_id', 'rating']].copy()
    test_pairs = test_pairs[test_pairs['book_id'].isin(book_to_idx.index)]

    pred_errors = []
    for row in test_pairs.itertuples(index=False):
        uid, bid, true_rating = row
        uidx = int(user_to_idx.loc[uid])
        bidx = int(book_to_idx.loc[bid])
        pred_errors.append((float(user_mean[uidx] + (user_factors[uidx] @ model.components_[:, bidx])), float(true_rating)))

    mae = mean_absolute_error([x[1] for x in pred_errors], [x[0] for x in pred_errors])
    rmse = np.sqrt(mean_squared_error([x[1] for x in pred_errors], [x[0] for x in pred_errors]))

    result = {'MAE': mae, 'RMSE': rmse}
    for k in k_values:
        metrics = []
        for i, uid in enumerate(eval_users):
            relevant = test_relevant.get(uid, set())
            if not relevant: continue
            top_idx = np.argpartition(-scores[i], k - 1)[:k]
            top_idx = top_idx[np.argsort(-scores[i, top_idx])]
            recommended = set(book_ids[top_idx])
            hits = [1 if book_ids[idx] in relevant else 0 for idx in top_idx]
            num_hits = sum(hits)
            precision = num_hits / k
            recall = num_hits / len(relevant)
            hit_rate = 1.0 if num_hits > 0 else 0.0
            dcg = sum(hit / math.log2(j + 2) for j, hit in enumerate(hits))
            ideal_len = min(k, len(relevant))
            idcg = sum(1 / math.log2(j + 2) for j in range(ideal_len))
            ndcg = dcg / idcg if idcg > 0 else 0.0
            metrics.append((precision, recall, hit_rate, ndcg))
        m = np.mean(metrics, axis=0)
        result[f'Precision@{k}'] = m[0]
        result[f'Recall@{k}'] = m[1]
        result[f'HitRate@{k}'] = m[2]
        result[f'NDCG@{k}'] = m[3]
    return result

## Исследование влияния гиперпараметра

Основным гиперпараметром является размерность разложения n_components.

Будут проверены значения 10, 20, 50, 100 и 200.

Чем больше размерность, тем больше скрытых факторов используется для описания матрицы пользователь-книга. Увеличение размерности может сделать восстановление матрицы точнее, но одновременно увеличивает время вычислений и риск переобучения.

In [6]:
results = []
models = {}
start_all = time.time()

for i, components in enumerate(GLOBAL_PARAMS['SVD_COMPONENTS'], 1):
    print(f'Запуск {i}/{len(GLOBAL_PARAMS["SVD_COMPONENTS"])}: n_components = {components}')
    start = time.time()
    model, user_factors, fit_time = fit_svd(components)
    metrics = evaluate_model(model, user_factors, components, GLOBAL_PARAMS['TOP_K_VALUES'])
    metrics['n_components'] = components
    metrics['FitTimeSec'] = fit_time
    results.append(metrics)
    models[components] = (model, user_factors)
    print(f'  завершено за {time.time() - start:.1f} сек.; NDCG@10 = {metrics["NDCG@10"]:.6f}; Precision@10 = {metrics["Precision@10"]:.6f}')

results_df = pd.DataFrame(results).set_index('n_components').sort_index()
display(results_df.round(6))
print(f'Общее время экспериментов: {time.time() - start_all:.1f} сек.')

Запуск 1/5: n_components = 10
  завершено за 0.7 сек.; NDCG@10 = 0.140460; Precision@10 = 0.106000
Запуск 2/5: n_components = 20
  завершено за 0.9 сек.; NDCG@10 = 0.130079; Precision@10 = 0.100500
Запуск 3/5: n_components = 50
  завершено за 1.5 сек.; NDCG@10 = 0.137904; Precision@10 = 0.105500
Запуск 4/5: n_components = 100
  завершено за 2.9 сек.; NDCG@10 = 0.122139; Precision@10 = 0.096500
Запуск 5/5: n_components = 200
  завершено за 5.5 сек.; NDCG@10 = 0.132114; Precision@10 = 0.101000


,MAE,RMSE,Precision@5,Recall@5,HitRate@5,NDCG@5,Precision@10,Recall@10,HitRate@10,NDCG@10,Precision@20,Recall@20,HitRate@20,NDCG@20,FitTimeSec
n_components,,,,,,,,,,,,,,,
10,0.661360,0.840656,0.159,0.056084,0.475,0.183582,0.1060,0.073089,0.535,0.140460,0.07075,0.097891,0.625,0.125828,0.538715
20,0.660563,0.840440,0.150,0.053521,0.480,0.168665,0.1005,0.069007,0.535,0.130079,0.07325,0.104078,0.670,0.123657,0.843488
50,0.659152,0.839947,0.133,0.046962,0.460,0.164568,0.1055,0.073113,0.580,0.137904,0.07375,0.101775,0.690,0.127023,1.449951
100,0.659361,0.840547,0.124,0.045098,0.475,0.145900,0.0965,0.067402,0.585,0.122139,0.07475,0.102465,0.740,0.119607,2.838164
200,0.659435,0.840434,0.137,0.047845,0.485,0.163487,0.1010,0.067363,0.590,0.132114,0.06825,0.089864,0.695,0.118908,5.405232


Общее время экспериментов: 11.7 сек.


## Выбор модели

Для основной сравнительной характеристики используется NDCG@10, поскольку эта метрика учитывает порядок рекомендаций.

Дополнительно анализируются Precision@10, Recall@10 и HitRate@10. Для оценки точности рейтингов используются RMSE и MAE.

Оптимальная размерность для ranking-задачи выбирается по максимальному NDCG@10. Для RMSE и MAE оптимальной считается размерность с минимальным значением соответствующей ошибки.

In [7]:
best_components = int(results_df['NDCG@10'].idxmax())
best_model, best_user_factors = models[best_components]

print(f'Размерность с максимальным NDCG@10: {best_components}')
display(results_df.loc[[best_components], ['MAE', 'RMSE', 'Precision@5', 'Recall@5', 'NDCG@5', 'HitRate@5', 'Precision@10', 'Recall@10', 'NDCG@10', 'HitRate@10', 'Precision@20', 'Recall@20', 'NDCG@20', 'HitRate@20', 'FitTimeSec']].round(6))

Размерность с максимальным NDCG@10: 10


,MAE,RMSE,Precision@5,Recall@5,NDCG@5,HitRate@5,Precision@10,Recall@10,NDCG@10,HitRate@10,Precision@20,Recall@20,NDCG@20,HitRate@20,FitTimeSec
n_components,,,,,,,,,,,,,,,
10,0.66136,0.840656,0.159,0.056084,0.183582,0.475,0.106,0.073089,0.14046,0.535,0.07075,0.097891,0.125828,0.625,0.538715


## Демонстрация на конкретном пользователе

Ниже показывается история наиболее высоких оценок выбранного пользователя и книги, которые модель рекомендует ему после исключения уже оценённых книг.

In [8]:
def get_recommendations(uid, model, user_factors, top_n=10):
    if uid not in user_to_idx.index: return pd.DataFrame()
    uidx = int(user_to_idx.loc[uid])
    scores = user_factors[uidx] @ model.components_ + user_mean[uidx]
    scores = scores.copy()
    seen = ui_matrix.getrow(uidx).indices
    scores[seen] = -np.inf
    top_idx = np.argsort(-scores)[:top_n]
    rec = pd.DataFrame({'book_id': book_ids[top_idx], 'predicted_rating': scores[top_idx]})
    cols = [c for c in ['book_id', 'title', 'authors', 'genres', 'average_rating', 'predicted_rating'] if c in books_df.columns or c == 'book_id']
    if 'book_id' not in books_df.columns: books_df['book_id'] = books_df.iloc[:, 0]
    return rec.merge(books_df[[c for c in cols if c in books_df.columns]], on='book_id', how='left')

example_user = int(sample_eval_users[0])
print(f'Пример пользователя: {example_user}')
print('Наиболее высокие оценки в обучающей выборке:')
display(train_df[train_df['user_id'] == example_user].sort_values('rating', ascending=False).head(10))
print('Рекомендации:')
display(get_recommendations(example_user, best_model, best_user_factors, 10))

Пример пользователя: 31916
Наиболее высокие оценки в обучающей выборке:


,user_id,book_id,rating
2786963,31916,39,5.0
2991151,31916,102,5.0
3121099,31916,4931,5.0
2786965,31916,1230,5.0
2787209,31916,229,5.0
2787009,31916,7086,5.0
2991150,31916,219,5.0
2787005,31916,546,5.0
2991161,31916,90,5.0
2991192,31916,250,5.0


Рекомендации:


,book_id,predicted_rating,title,authors,genres,average_rating
0,4,4.961335,To Kill a Mockingbird,['Harper Lee'],"['classics', 'fiction', 'historical-fiction', ...",4.25
1,54,4.749928,The Hitchhiker's Guide to the Galaxy (Hitchhik...,['Douglas Adams'],"['science-fiction', 'fiction', 'fantasy', 'cla...",4.20
2,65,4.677536,Slaughterhouse-Five,['Kurt Vonnegut Jr.'],"['classics', 'fiction', 'science-fiction', 'hi...",4.06
3,55,4.657750,Brave New World,['Aldous Huxley'],"['classics', 'fiction', 'science-fiction', 'fa...",3.97
4,70,4.639680,"Ender's Game (Ender's Saga, #1)",['Orson Scott Card'],"['science-fiction', 'fiction', 'young-adult', ...",4.30
5,129,4.622576,One Flew Over the Cuckoo's Nest,['Ken Kesey'],"['classics', 'fiction', 'psychology', 'contemp...",4.18
6,113,4.619671,Catch-22,['Joseph Heller'],"['classics', 'fiction', 'historical-fiction']",3.98
7,19,4.616089,The Fellowship of the Ring (The Lord of the Ri...,"['[J.R.R. Tolkien', 'Francis Ledoux', 'Maria S...","['fantasy', 'classics', 'fiction', 'young-adul...",4.34
8,135,4.606212,"A Storm of Swords (A Song of Ice and Fire, #3)",['George R.R. Martin'],"['fantasy', 'fiction']",4.54
9,80,4.604465,The Little Prince,"['Antoine de Saint-Exupéry', 'Richard Howard',...","['classics', 'fiction', 'fantasy', 'young-adul...",4.28


## Сравнение с предыдущим User-Based + Item-Based KNN

Предыдущий гибридный подход показал лучшие метрики при K = 10.

В следующей ячейке эти значения сравниваются с лучшим вариантом SVD (n_components = 10) по тем же четырём ranking-метрикам.

In [21]:
knn_metrics = pd.DataFrame({
    'Precision@10': [0.210606],
    'Recall@10': [0.144350],
    'NDCG@10': [0.259965],
    'HitRate@10': [0.833333]
}, index=['User+Item KNN'])

svd_comparison = results_df.loc[[best_components],['Precision@10', 'Recall@10', 'NDCG@10', 'HitRate@10']].copy()
svd_comparison.index = [f'SVD ({i})' for i in svd_comparison.index]
comparison_df = pd.concat([knn_metrics, svd_comparison])
display(comparison_df.round(6))

,Precision@10,Recall@10,NDCG@10,HitRate@10
User+Item KNN,0.210606,0.144350,0.259965,0.833333
SVD (10),0.106000,0.073089,0.140460,0.535000


## Интерпретация результатов

В проведённом эксперименте User-Based + Item-Based KNN показал более высокое качество по всем четырём ranking-метрикам, чем SVD. Особенно заметно преимущество KNN по HitRate@10 и NDCG@10, то есть он чаще находил релевантные книги и лучше располагал их в начале списка.

Для SVD лучший результат по основным ranking-метрикам получен при n_components = 10. При дальнейшем увеличении размерности устойчивого улучшения качества не наблюдается. Значения Precision@10, Recall@10 и NDCG@10 изменяются немонотонно, а при некоторых размерностях даже снижаются.

При этом HitRate@10 постепенно растёт вместе с размерностью и достигает максимума при n_components = 200. Это означает, что увеличение числа скрытых факторов позволяет немного чаще находить хотя бы одну релевантную книгу, но не повышает качество всего списка рекомендаций.

Таким образом, увеличение размерности скрытого пространства не гарантирует улучшения SVD. В данном эксперименте небольшая модель с n_components = 10 оказалась наиболее удачной по основным метрикам ранжирования.

## Сравнение с KNN

Предыдущий гибридный KNN превосходит SVD по Precision@10, Recall@10, NDCG@10 и HitRate@10. Следовательно, на данном датасете и при выбранном способе тестирования KNN лучше подходит для формирования Top-10 рекомендаций.

SVD при этом использует компактное скрытое представление пользователей и книг, что делает метод удобным для работы с матрицами взаимодействий. Однако в текущей конфигурации это представление хуже сохраняет информацию, необходимую для точного ранжирования книг.

## Выводы

SVD рассматривает матрицу пользователь-книга как результат взаимодействия скрытых факторов пользователей и книг. Размерность разложения определяет количество таких факторов.

В эксперименте сравниваются несколько значений n_components: 10, 20, 50, 100 и 200. Для каждого значения рассчитываются RMSE, MAE, Precision@5, Recall@5, NDCG@5, HitRate@5, Precision@10, Recall@10, NDCG@10, HitRate@10, Precision@20, Recall@20, NDCG@20 и HitRate@20.

Основным критерием выбора модели служит NDCG@10, поскольку для рекомендательной системы важно не только наличие релевантных книг среди рекомендаций, но и их положение в начале списка.

Эксперимент показал, что увеличение размерности не приводит к стабильному росту качества. Лучший вариант SVD по основным ranking-метрикам получен при n_components = 10.

По сравнению с предыдущим User-Based + Item-Based KNN метод SVD показал более низкое качество рекомендаций по всем основным метрикам. Поэтому в рамках проведённого эксперимента KNN оказался более эффективным для данной задачи, тогда как SVD представляет собой альтернативный подход на основе скрытых факторов.